# Oscillatory DTB through `DTB_Ver3`

[Open in Colab](https://colab.research.google.com/github/sun-mengwei/dtb-colab-experiments/blob/codex%2Fgame-dynamics-dtb/DTB_Ver3/notebooks/oscillatory_accumulated_map_dtb.ipynb)

This is the package-based counterpart of [`oscillatory_accumulated_map_parameter_evolving_dtb_mmnn.ipynb`](https://github.com/sun-mengwei/dtb-colab-experiments/blob/codex/game-dynamics-dtb/DTB_Game_Ver2/oscillatory_accumulated_map_parameter_evolving_dtb_mmnn.ipynb). It uses the reusable game, experiment, MMNN, tangent projection, reference, serialization, and plotting interfaces from `DTB_Ver3`.


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

# Find a local checkout. In Colab, obtain the same branch automatically.
candidates = [Path.cwd(), *Path.cwd().parents]
repo_root = next((path for path in candidates if (path / 'DTB_Ver3').is_dir()), None)
if repo_root is None:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Run inside the repository or upload the DTB_Ver3 folder.')
    repo_root = Path('/content/dtb-colab-experiments')
    if not repo_root.exists():
        subprocess.run([
            'git', 'clone', '--depth', '1', '--branch', 'codex/game-dynamics-dtb',
            'https://github.com/sun-mengwei/dtb-colab-experiments.git', str(repo_root),
        ], check=True)
sys.path.insert(0, str(repo_root.resolve()))

from DTB_Ver3 import ExperimentConfig, OscillatoryGame, ResidualMLP, ResidualMMNN, run_experiment
from DTB_Ver3.utils import plot_configuration_diagnostics

package_dir = repo_root / 'DTB_Ver3'


## Game, neural-family choice, and DTB update

The two-dimensional potential is

\[
\Phi(x)=-\frac{\lambda}{2}(x_1^2+x_2^2)
-\frac{\gamma}{2}(x_1-x_2)^2
+\frac{\epsilon}{\omega}\{\cos(\omega x_1)+\cos(\omega x_2)\},
\]

with \(\lambda=0.5\), \(\gamma=0.2\), \(\epsilon=0.5\), and \(\omega=4\pi\). The deterministic game velocity is \(g(x)=\nabla\Phi(x)\).

Set `NETWORK_FAMILY` to `"mmnn"` (the default matching experiment) or `"mlp"`. Both choices use a residual map initialized exactly as the identity. For MMNN, each layer is \(A\,\sigma(Wx+b)+c\): the random features \(W,b\) are frozen and only \(A,c\) belong to \(\theta\). `rank` is used only by MMNN; it has no role in an MLP.

The labels \(z_i\sim U([-1,1]^2)\) stay fixed. Every trainable parameter coordinate is used, and the full tangent matrix is recomputed at the current \(\theta_k\) on the labels at every step:

\[
\alpha_k=\arg\min_\alpha\|J_{\theta_k}(z)\alpha-g(X_k)\|_2^2,
\]
\[
X_{k+1}=X_k+hJ_{\theta_k}(z)\alpha_k,\qquad
\theta_{k+1}=\theta_k+h\alpha_k.
\]

The physical state remains the DTB state \(X_k\); it is never replaced by \(T_{\theta_{k+1}}(z)\). The explicit-Euler reference starts from the same labels and uses the same step size.


In [ ]:
SEED = 2026
N_PARTICLES = 1000
T_FINAL = 1.0
H = 0.005

# Flexible neural choice. Rank appears only in the MMNN branch.
NETWORK_FAMILY = 'mmnn'  # choose 'mmnn' or 'mlp'
MLP_WIDTH, MLP_DEPTH = 16, 2
MMNN_WIDTH, MMNN_RANK, MMNN_DEPTH = 12, 12, 3
ACTIVATION = 'tanh'

# Tangent bundle size: None uses every trainable coordinate.
# Set an integer such as 32, 64, or 128 to use a smaller subset.
TANGENT_BASIS_SIZE = None
SUBSET_TANGENT_SELECTION = 'fixed'  # or 'resample_each_step'

SVD_RTOL = 1e-3
JACOBIAN_CHUNK = 128

LAMBDA = 0.5
GAMMA = 0.2
EPSILON = 0.5
OMEGA = 4 * np.pi

game = OscillatoryGame(
    linear_damping=LAMBDA,
    coupling=GAMMA,
    epsilon=EPSILON,
    omega=OMEGA,
)

# Reproduce the source notebooks' RNG order: sample z with the global generator
# before initializing the chosen neural map. The experiment independently draws
# the identical z from its private generator.
torch.manual_seed(SEED)
_ = torch.rand((N_PARTICLES, game.dim), dtype=torch.float32)

if NETWORK_FAMILY == 'mmnn':
    model_kind = 'residual_mmnn'
    network_config = dict(
        width=MMNN_WIDTH,
        rank=MMNN_RANK,
        depth=MMNN_DEPTH,
    )
    model = ResidualMMNN(
        game.dim,
        activation=ACTIVATION,
        dtype=torch.float32,
        zero_init_output=True,
        **network_config,
    )
elif NETWORK_FAMILY == 'mlp':
    model_kind = 'residual_mlp'
    network_config = dict(width=MLP_WIDTH, depth=MLP_DEPTH)
    model = ResidualMLP(
        game.dim,
        activation=ACTIVATION,
        dtype=torch.float32,
        zero_init_output=True,
        **network_config,
    )
else:
    raise ValueError("NETWORK_FAMILY must be 'mmnn' or 'mlp'.")

TRAINABLE_PARAMETER_COUNT = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)
FROZEN_PARAMETER_COUNT = sum(
    parameter.numel() for parameter in model.parameters() if not parameter.requires_grad
)
print({
    'network_family': NETWORK_FAMILY,
    'model_kind': model_kind,
    'trainable_parameters': TRAINABLE_PARAMETER_COUNT,
    'frozen_parameters': FROZEN_PARAMETER_COUNT,
    **network_config,
})

config = ExperimentConfig(
    dynamics='deterministic',
    run_reference=True,
    particle_count=N_PARTICLES,
    initial_law='uniform',
    initial_low=-1.0,
    initial_high=1.0,
    step_size=H,
    final_time=T_FINAL,
    snapshot_times=(0.0, 0.25, 0.5, 0.75, 1.0),
    activation=ACTIVATION,
    model_kind=model_kind,
    zero_init_output=True,
    basis_size=(
        TRAINABLE_PARAMETER_COUNT
        if TANGENT_BASIS_SIZE is None
        else TANGENT_BASIS_SIZE
    ),
    subset_tangent_selection=SUBSET_TANGENT_SELECTION,
    tangent_input_mode='fixed_initial_labels',
    track_network_map=True,
    svd_rtol=SVD_RTOL,
    jacobian_chunk=JACOBIAN_CHUNK,
    seed=SEED,
    dtype='float32',
    device='auto',
    progress_reports=5,
    output_dir=package_dir / 'results' / f'oscillatory_dtb_{NETWORK_FAMILY}',
    **network_config,
)

# Sanity check the packaged velocity against grad(Phi).
check_x = torch.tensor([[0.17, -0.31]], dtype=torch.float32, requires_grad=True)
check_gradient = torch.autograd.grad(game.potential(check_x).sum(), check_x)[0]
assert torch.allclose(check_gradient, game.velocity(check_x), atol=2e-6, rtol=0)


## Run the package experiment

`result.dtb_snapshots` contains the physical DTB state. Optional network tracking records the distinct nonlinear map `T_theta(z)` and its RMS gap from DTB.


In [ ]:
result = run_experiment(game, config, model=model)

print(
    f"Tangent coordinates used: {result.selected_indices.size} / "
    f"{TRAINABLE_PARAMETER_COUNT}; selection={config.subset_tangent_selection}"
)

summary = result.summary()
print('Summary:')
for key, value in summary.items():
    print(f'  {key}: {value}')
print(f"mean/max relative projection error: "
      f"{result.relative_projection_error.mean():.3e} / "
      f"{result.relative_projection_error.max():.3e}")


## DTB, neural map, and Euler reference


In [ ]:
snapshot_times = sorted(result.dtb_snapshots)
grid_axis = torch.linspace(-1.05, 1.05, 140)
grid_x1, grid_x2 = torch.meshgrid(grid_axis, grid_axis, indexing='xy')
grid_points = torch.stack((grid_x1.ravel(), grid_x2.ravel()), dim=-1)
potential_grid = game.potential(grid_points).reshape(grid_x1.shape).numpy()

rows = (
    (result.dtb_snapshots, 'DTB', '#176b87'),
    (result.network_snapshots, r'actual $T_{\theta_k}(z)$', '#7c3aed'),
    (result.reference_snapshots, 'explicit Euler', '#b45309'),
)
fig, axes = plt.subplots(
    len(rows), len(snapshot_times),
    figsize=(3.0 * len(snapshot_times), 8.2),
    sharex=True, sharey=True, layout='constrained',
)
for column, time_value in enumerate(snapshot_times):
    for row, (snapshots, label, color) in enumerate(rows):
        axis = axes[row, column]
        axis.contour(grid_x1, grid_x2, potential_grid, levels=16,
                     linewidths=0.45, alpha=0.55)
        points = snapshots[time_value]
        axis.scatter(points[:, 0], points[:, 1], s=7, alpha=0.5,
                     color=color, linewidths=0, rasterized=True)
        axis.set_aspect('equal')
        axis.grid(alpha=0.2)
        if column == 0:
            axis.set_ylabel(label)
        if row == 0:
            axis.set_title(f't = {time_value:g}')
        if row == len(rows) - 1:
            axis.set_xlabel(r'$x_1$')
fig.suptitle('Oscillatory game: DTB, evolving network, and Euler')
comparison_path = result.output_dir / 'oscillatory_map_comparison.png'
fig.savefig(comparison_path, dpi=300, bbox_inches='tight')
plt.show()
print('Saved:', comparison_path)


## Time-dependent diagnostics

The package records the DTB trajectory RMS against Euler, relative projection residual, coefficient norm, singular values, retained rank, and the gap between the DTB state and the neural map.


In [ ]:
plot_configuration_diagnostics(
    result.times,
    result.trajectory_rms_error,
    result.projection_times,
    result.relative_projection_error,
    result.alpha_norm,
    result.jacobian_condition,
    output_path=result.output_dir / 'configuration_diagnostics.png',
)
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(15, 7.5), layout='constrained')
axes[0, 0].semilogy(result.times, result.trajectory_rms_error + 1e-15,
                     label='DTB')
axes[0, 0].semilogy(result.times, result.network_trajectory_rms_error + 1e-15,
                     label=r'actual $T_{\theta_k}(z)$')
axes[0, 0].set_title('RMS error against Euler')
axes[0, 0].legend()

axes[0, 1].semilogy(result.projection_times,
                     result.relative_projection_error + 1e-15)
axes[0, 1].set_title(r'$\|J_k\alpha_k-g_k\|_2/\|g_k\|_2$')

axes[0, 2].semilogy(result.times, result.physical_network_gap + 1e-15,
                     label=r'$\|X_k-T_{\theta_k}(z)\|_{RMS}$')
axes[0, 2].set_title('DTB/network gap')
axes[0, 2].legend()

axes[1, 0].semilogy(result.projection_times, result.alpha_norm + 1e-15)
axes[1, 0].set_title(r'$\|\alpha_k\|_2$')
axes[1, 1].semilogy(result.projection_times, result.jacobian_sigma_max + 1e-15,
                     label=r'$\sigma_{max}$')
axes[1, 1].semilogy(result.projection_times, result.jacobian_sigma_min + 1e-15,
                     label=r'$\sigma_{min,\mathrm{retained}}$')
axes[1, 1].set_title('Selected tangent singular values')
axes[1, 1].legend()
axes[1, 2].plot(result.projection_times, result.retained_rank)
axes[1, 2].set_title('Retained tangent rank')
for axis in axes.ravel():
    axis.set_xlabel('time')
    axis.grid(alpha=0.25, which='both')
diagnostic_path = result.output_dir / 'oscillatory_diagnostics.png'
fig.savefig(diagnostic_path, dpi=300, bbox_inches='tight')
plt.show()

# Mean potential on the stored clouds.
def snapshot_potential(snapshots):
    return np.asarray([
        game.potential(torch.from_numpy(snapshots[t])).mean().item()
        for t in snapshot_times
    ])

fig, axis = plt.subplots(figsize=(7, 4.2), layout='constrained')
axis.plot(snapshot_times, snapshot_potential(result.reference_snapshots), 'o-',
          label='Euler')
axis.plot(snapshot_times, snapshot_potential(result.dtb_snapshots), 'o-',
          label='DTB')
axis.plot(snapshot_times, snapshot_potential(result.network_snapshots), 'o-',
          label=r'actual $T_{\theta_k}(z)$')
axis.set(xlabel='time', ylabel='mean potential', title='Mean oscillatory potential')
axis.grid(alpha=0.25)
axis.legend()
potential_path = result.output_dir / 'mean_potential.png'
fig.savefig(potential_path, dpi=300, bbox_inches='tight')
plt.show()

print('Saved:', diagnostic_path)
print('Saved:', potential_path)


## Package the complete output for download


In [ ]:
archive = shutil.make_archive(
    str(result.output_dir),
    'zip',
    root_dir=result.output_dir.parent,
    base_dir=result.output_dir.name,
)
print('Saved result archive:', archive)

try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(archive)
